In [ ]:
# f=open(r'..\data\raw\parking.csv','r') # 상태경로, ..두개는 상위 경로 의미


In [28]:
import pandas as pd
import numpy as np
import pymysql
import os

from dotenv import load_dotenv
load_dotenv()

conn = pymysql.connect(
    host=os.environ.get('DB_HOST', '127.0.0.1'), # 데이터베이스 서버(컴퓨터) 주소 (localhost, 127.0.0.1)
    port=int(os.environ.get('DB_PORT', '3306')),
    user=os.environ.get('DB_USER', 'analyst'),
    password=os.environ.get('DB_PASSWORD', ''),
    database=os.environ.get('DB_NAME', 'project_db'),
    charset='utf8mb4'
)

DB_CONFIG = {
    'host': os.environ.get('DB_HOST', '127.0.0.1'),
    'port': int(os.environ.get('DB_PORT', '3306')),
    'user': os.environ.get('DB_USER', 'analyst'),
    'password': os.environ.get('DB_PASSWORD', ''),
    'database': os.environ.get('DB_NAME', 'project_db'),
    'charset': 'utf8mb4',
    'autocommit': False,
}

In [ ]:
parking_df=pd.read_csv('../data/raw/parking.csv',encoding='cp949')
parking_df.head(3)
parking_df.columns

C:\Users\kevin\AppData\Local\Temp\ipykernel_34956\3058404900.py:2: DtypeWarning: Columns (0: 추가단위요금, 1: 제공기관코드) have mixed types. Specify dtype option on import or set low_memory=False.
  parking_df=pd.read_csv('../data/raw/parking.csv',encoding='cp949')


Index(['주차장관리번호', '주차장명', '주차장구분', '주차장유형', '소재지도로명주소', '소재지지번주소', '주차구획수',
       '급지구분', '부제시행구분', '운영요일', '평일운영시작시각', '평일운영종료시각', '토요일운영시작시각',
       '토요일운영종료시각', '공휴일운영시작시각', '공휴일운영종료시각', '요금정보', '주차기본시간', '주차기본요금',
       '추가단위시간', '추가단위요금', '1일주차권요금적용시간', '1일주차권요금', '월정기권요금', '결제방법', '특기사항',
       '관리기관명', '전화번호', '위도', '경도', '장애인전용주차구역보유여부', '데이터기준일자', '제공기관코드',
       '제공기관명'],
      dtype='str')

In [10]:
parking_df.describe()

,주차구획수,주차기본시간,주차기본요금,추가단위시간,1일주차권요금적용시간,1일주차권요금,월정기권요금,위도,경도
count,18879.000000,18879.000000,13579.000000,7886.000000,4.378000e+03,5092.000000,5145.000000,18118.000000,18118.000000
mean,56.462683,47.857302,413.463068,7.808648,4.760653e+04,4749.873723,48289.282799,36.154751,127.614937
std,97.586807,220.736429,2893.068876,13.944864,6.148923e+05,5871.518993,44701.485053,1.247927,0.914373
min,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,26.128492,124.619641
25%,14.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,35.293417,126.851979
50%,29.000000,0.000000,0.000000,5.000000,5.000000e+00,4700.000000,50000.000000,36.122137,127.355084
75%,60.500000,30.000000,400.000000,10.000000,2.400000e+01,7000.000000,70000.000000,37.347234,128.469245
max,4261.000000,1440.000000,40000.000000,500.000000,8.002000e+06,70000.000000,350000.000000,38.543996,130.909983


In [15]:
car_df=pd.read_csv('../data/raw/car.csv',encoding='cp949')
car_df.head(3)
car_df.columns

Index(['시도명(1)', '시군구(1)', '2026.06'], dtype='str')

In [21]:
parking_df_mod=parking_df[['소재지지번주소','주차구획수','요금정보']]
parking_df_mod.head(3)

,소재지지번주소,주차구획수,요금정보
0,전라남도 광양시 옥룡면 운평리 262,10,무료
1,전라남도 광양시 금호동 624-9,10,무료
2,전라남도 광양시 태인동 398,11,무료


In [ ]:
import pandas as pd
import numpy as np


# 주요 분석사항: 자동차등록대수 대비 주차구획 공급량과 요금 부담이 적절한가
# 세부 분석사항 
# 1. 시군구별 자동차등록대수 대비 주차구획 공급량 비교
# 2. 높은 유료주차장 비율과 낮은 주차구획 공급량 지역 탐색
# 정책 개선사항 제시: 주차 부담을 줄이기 위해 주차구획 추가 공급이 필요한 지역과 요금 인하가 필요한 지역 탐색

# 1. 데이터 불러오기
car_df = pd.read_csv('../data/raw/car.csv', encoding='cp949')
parking_df = pd.read_csv('../data/raw/parking.csv', encoding='cp949')

# 2. parking.csv 행정구역 정제 (도로명주소와 지번주소 열간 통합)
parking_df['address'] = parking_df['소재지도로명주소'].fillna(parking_df['소재지지번주소'])

# 확인된 오타 교정
parking_df['address'] = parking_df['address'].str.replace('경기도동두천시평화로', '경기도 동두천시 평화로')
parking_df['address'] = parking_df['address'].str.replace('강원특별자치도강릉시', '강원특별자치도 강릉시')
parking_df['address'] = parking_df['address'].str.replace('경상남동', '경상남도')
parking_df['address'] = parking_df['address'].str.replace('전븍특별자치도', '전북특별자치도')

# 시도, 시군구 추출
parking_df['시도_raw'] = parking_df['address'].str.split().str[0]
parking_df['시군구_raw'] = parking_df['address'].str.split().str[1]

# car.csv의 시도명과 매칭되도록 시도명칭 통일
sido_map = {
    '서울특별시': '서울', '부산광역시': '부산', '대구광역시': '대구', '인천광역시': '인천',
    '광주광역시': '광주', '대전광역시': '대전', '울산광역시': '울산', '세종특별자치시': '세종',
    '경기도': '경기', '강원특별자치도': '강원', '충청북도': '충북', '충청남도': '충남',
    '전라북도': '전북', '전북특별자치도': '전북', '전라남도': '전남', '경상북도': '경북',
    '경상남도': '경남', '제주특별자치도': '제주'
}
parking_df['시도'] = parking_df['시도_raw'].map(sido_map)

# 3. 기준 키 컬럼('시도시군구') 생성 
# car.csv : 시도명(1) + 시군구(1)
# parking.csv : 시도 + 시군구_raw
car_df['시도시군구'] = car_df['시도명(1)'] + " " + car_df['시군구(1)']
parking_df['시도시군구'] = parking_df['시도'] + " " + parking_df['시군구_raw']

# 4. 유료지표 생성 (1='유료', 0='무료', 0.5='혼합')
fee_map = {'유료': 1.0, '무료': 0.0, '혼합': 0.5, '유료+무': 1.0}
parking_df['유료지표'] = parking_df['요금정보'].map(fee_map)

# 5. 주차장 데이터 그룹화 연산
parking_summary = parking_df.groupby('시도시군구').agg(
    주차구획수_합계=('주차구획수', 'sum'),
    유료주차장수_평균=('유료지표', 'mean')
).reset_index()

# 6. 자동차 데이터와 병합 (생성한 '시도시군구' 기준)
merged_data = pd.merge(parking_summary, car_df, on='시도시군구', how='inner')

# 7. 주차구획수 공급률 도출 (주차구획수 합계 / 2026.06 기준 자동차등록대수)
# 자동차 등록대수 column = '2026.06'
merged_data['주차구획_공급률'] = np.round(merged_data['주차구획수_합계'] / merged_data['2026.06'], 2)
merged_data['자동차등록대수'] = merged_data['2026.06']
# 8. 병합된 데이터 표시
result = merged_data[['시도시군구', '주차구획수_합계', '유료주차장수_평균', '자동차등록대수', '주차구획_공급률']]
# print(result)

# 9. 정렬 및 결과 확인
result_supply_asc = result.sort_values(by='주차구획_공급률', ascending=True)
result_supply_desc = result.sort_values(by='주차구획_공급률', ascending=False)
result_fee_desc = result.sort_values(by=['유료주차장수_평균','주차구획_공급률'], ascending=[False,True])
# print(result_supply_asc.head(10))
# print(result_supply_desc.head(10))
print(result_fee_desc.head(10))

       시도시군구  주차구획수_합계  유료주차장수_평균  자동차등록대수  주차구획_공급률
104  부산 부산진구       439        1.0   179936      0.00
109   부산 수영구       413        1.0   125995      0.00
129   서울 성동구        31        1.0   105145      0.00
41    경남 거제시      1672        1.0   115141      0.01
51    경남 진주시      1931        1.0   182571      0.01
122   서울 노원구      1236        1.0   148896      0.01
125   서울 동작구       821        1.0   104863      0.01
131   서울 송파구      3133        1.0   243634      0.01
135   서울 은평구      1734        1.0   132320      0.01
150    인천 서구      2234        1.0   338754      0.01


C:\Users\kevin\AppData\Local\Temp\ipykernel_34956\4240707791.py:13: DtypeWarning: Columns (0: 추가단위요금, 1: 제공기관코드) have mixed types. Specify dtype option on import or set low_memory=False.
  parking_df = pd.read_csv('../data/raw/parking.csv', encoding='cp949')
